<a href="https://colab.research.google.com/github/SeRJ02/Price-Comparison/blob/main/EarnKaro_Price_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 EarnKaro Price Comparison Telegram Bot

**What it does:** Send a product link from Amazon/Flipkart/Myntra/Hamara Mall → bot compares prices across all platforms and shows EarnKaro commission.

---

## 📋 Run Order
1. **Cell 1** — Install dependencies
2. **Cell 2** — Mount Google Drive & set working directory
3. **Cell 3** — Write all source files
4. **Cell 4** — Upload service_account.json (for Google Sheets)
5. **Cell 5** — Enter Telegram bot token
6. **Cell 6** — Run unit tests
7. **Cell 7** — Keep-alive (prevents Colab timeout)
8. **Cell 8** — **Start the bot** (blocks — must be last)

---

## 🔄 Session Restart?
If Colab disconnects, run cells **1 → 2 → 5 → 8** (skip 3 if files are still on Drive).

In [3]:
# === Cell 1: Install Dependencies ===
!pip install requests==2.31.0 beautifulsoup4==4.12.2 lxml==4.9.3 rapidfuzz==3.5.2 \
    python-telegram-bot==20.6 gspread==5.12.0 google-auth==2.23.4 fake-useragent==1.4.0 -q
print("✅ Dependencies installed")

✅ Dependencies installed


In [4]:
# === Cell 2: Mount Google Drive & Set Working Directory ===
import os
from google.colab import drive

drive.mount("/content/drive")

BOT_DIR = "/content/drive/MyDrive/earnkaro_bot"
os.makedirs(BOT_DIR, exist_ok=True)
os.chdir(BOT_DIR)

# Add to Python path so imports work
import sys
if BOT_DIR not in sys.path:
    sys.path.insert(0, BOT_DIR)

print(f"✅ Working directory: {os.getcwd()}")

Mounted at /content/drive
✅ Working directory: /content/drive/MyDrive/earnkaro_bot


In [5]:
# === Cell 3: Write All Source Files to Drive ===
# This writes all .py modules to your Drive so imports work.
# Only needs to run once (or after code changes).

import os

files = {}

files["config.py"] = '''# config.py — All constants, headers, CSS selectors

# Platform domain detection
PLATFORM_DOMAINS = {
    "amazon.in": "amazon",
    "flipkart.com": "flipkart",
    "myntra.com": "myntra",
    "hamaramall.com": "hamaramall",
}

# Request headers per platform
HEADERS = {
    "amazon": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-IN,en;q=0.9",
    },
    "flipkart": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    },
    "myntra": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    },
    "hamaramall": {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    },
}

# Matching score thresholds
SCORE_EXACT = 75      # Show as ✅ Exact match
SCORE_SIMILAR = 40    # Show as ⚠️ Similar product
# Below 40 = ❌ Not found

# Search result count to evaluate per platform
TOP_N_RESULTS = 5

# Delays between requests (seconds) to avoid rate limiting
REQUEST_DELAY = 1.5

# Google Sheet config
SHEET_NAME = "EarnKaro Commissions"
SHEET_TAB = "Category Commission"
COMMISSION_COL_CATEGORY = 0  # Column A
COMMISSION_COL_PERCENT = 1   # Column B

# CSS Selectors — Amazon
AMAZON_SELECTORS = {
    "brand": "#bylineInfo span",
    "title": "#productTitle",
    "price_primary": "#priceblock_ourprice",
    "price_whole": ".a-price-whole",
    "price_deal": "#priceblock_dealprice",
    "breadcrumb": "#wayfinding-breadcrumbs_feature_div",
    "search_result": 'div[data-component-type="s-search-result"]',
    "search_title": ".a-size-medium, .a-size-base-plus",
    "search_price": ".a-price-whole",
    "search_link": "a.a-link-normal",
}

# CSS Selectors — Flipkart
FLIPKART_SELECTORS = {
    "title": "._35KyD6, .B_NuCI",
    "price": "._30jeq3._16Jk6d",
    "breadcrumb": "._2whKao",
}

# CSS Selectors — Myntra
MYNTRA_SELECTORS = {
    "brand": ".pdp-title",
    "title": ".pdp-name",
    "price": ".pdp-price strong, .pdp-mrp",
}

# CSS Selectors — Hamara Mall (to be updated after live inspection)
HAMARAMALL_SELECTORS = {
    "brand": "",
    "title": "",
    "price": "",
    "category": "",
}

# Platform emoji map
PLATFORM_EMOJI = {
    "amazon": "🟠",
    "flipkart": "🟡",
    "myntra": "🔵",
    "hamaramall": "🏪",
}

# Stop words for keyword extraction
STOP_WORDS = {"for", "with", "and", "of", "the", "in", "a", "an", "-"}
'''

files["utils.py"] = '''# utils.py — Shared helpers

import re
from urllib.parse import urlparse
from config import PLATFORM_DOMAINS, STOP_WORDS


def detect_platform(url: str) -> str:
    """Parse URL domain and return platform name. Raises ValueError if unknown."""
    domain = urlparse(url).netloc.lower().replace("www.", "")
    for key, platform in PLATFORM_DOMAINS.items():
        if key in domain:
            return platform
    raise ValueError(f"Unrecognised domain: {domain}")


def normalize_quantity(text: str) -> str:
    """Standardize quantity strings for comparison."""
    if not text:
        return ""
    text = text.lower().strip().replace(" ", "")

    # Convert kg to g
    kg_match = re.match(r"^(\\d+(?:\\.\\d+)?)kg$", text)
    if kg_match:
        grams = int(float(kg_match.group(1)) * 1000)
        return f"{grams}g"

    # Normalize "gm" to "g"
    text = re.sub(r"(\\d)gm$", r"\\1g", text)

    # Normalize "pack of N" / "xN" to "Npack"
    pack_match = re.match(r"packof(\\d+)", text)
    if pack_match:
        return f"{pack_match.group(1)}pack"
    x_match = re.match(r"x(\\d+)$", text)
    if x_match:
        return f"{x_match.group(1)}pack"

    return text


def extract_quantity_from_title(title: str) -> str:
    """Use regex to find quantity patterns in a product title."""
    if not title:
        return ""
    patterns = [
        r"(\\d+(?:\\.\\d+)?\\s*(?:ml|l|kg|g|gm|oz))\\b",  # 250ml, 1.5kg, 200g, 100gm
        r"(pack\\s*of\\s*\\d+)",                           # pack of 2
        r"\\b(x\\d+)\\b",                                  # x3
    ]
    for pattern in patterns:
        match = re.search(pattern, title, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return ""


def clean_price(price_str: str) -> int:
    """Strip currency symbols, commas, decimals. Return integer price or 0."""
    if not price_str:
        return 0
    # Find the first number (with optional decimals)
    match = re.search(r"(\\d[\\d,]*(?:\\.\\d+)?)", price_str)
    if not match:
        return 0
    cleaned = match.group(1).replace(",", "")
    try:
        return int(float(cleaned))
    except ValueError:
        return 0


def extract_keywords(title: str, brand: str) -> list:
    """Extract meaningful keywords from title, removing brand, stop words, quantity."""
    if not title:
        return []
    # Remove brand from title
    cleaned = re.sub(re.escape(brand), "", title, flags=re.IGNORECASE).strip()
    # Remove quantity strings
    qty = extract_quantity_from_title(cleaned)
    if qty:
        cleaned = cleaned.replace(qty, "")
    # Tokenize, lowercase, filter
    words = cleaned.lower().split()
    keywords = [w.strip(",-()") for w in words if w.strip(",-()") and w.strip(",-()") not in STOP_WORDS]
    return keywords
'''

files["extractor.py"] = '''# extractor.py — Source page scraper

import time
import requests
from bs4 import BeautifulSoup

from config import HEADERS, AMAZON_SELECTORS, FLIPKART_SELECTORS, MYNTRA_SELECTORS, HAMARAMALL_SELECTORS
from utils import detect_platform, extract_quantity_from_title, extract_keywords, clean_price


def extract_product(url: str) -> dict:
    """Main entry: fetch URL, detect platform, extract structured product data."""
    product = {
        "platform": "",
        "url": url,
        "brand": "",
        "name": "",
        "quantity": "",
        "keywords": [],
        "category": "",
        "price": 0,
        "search_query": "",
        "warnings": [],
    }

    try:
        platform = detect_platform(url)
        product["platform"] = platform

        headers = HEADERS.get(platform, {})
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")

        extractors = {
            "amazon": extract_from_amazon,
            "flipkart": extract_from_flipkart,
            "myntra": extract_from_myntra,
            "hamaramall": extract_from_hamaramall,
        }

        extractor = extractors.get(platform)
        if extractor:
            data = extractor(soup, url)
            product.update(data)

        # Fill derived fields
        if product["name"] and not product["quantity"]:
            product["quantity"] = extract_quantity_from_title(product["name"])
        if product["name"] and product["brand"]:
            product["keywords"] = extract_keywords(product["name"], product["brand"])
        product["search_query"] = build_search_query(product)

    except Exception as e:
        product["warnings"].append(f"Extraction error: {str(e)}")

    return product


def extract_from_amazon(soup, url) -> dict:
    """Scrape an Amazon.in product page."""
    data = {"warnings": []}

    # Brand
    try:
        brand_el = soup.select_one(AMAZON_SELECTORS["brand"])
        if brand_el:
            data["brand"] = brand_el.get_text(strip=True)
        else:
            # Fallback: first word of title
            title_el = soup.select_one(AMAZON_SELECTORS["title"])
            if title_el:
                data["brand"] = title_el.get_text(strip=True).split()[0]
    except Exception:
        data["warnings"].append("brand selector failed")

    # Title
    try:
        title_el = soup.select_one(AMAZON_SELECTORS["title"])
        data["name"] = title_el.get_text(strip=True) if title_el else ""
    except Exception:
        data["warnings"].append("title selector failed")

    # Price — try multiple selectors in order
    try:
        price_text = ""
        for sel_key in ["price_primary", "price_whole", "price_deal"]:
            el = soup.select_one(AMAZON_SELECTORS[sel_key])
            if el:
                price_text = el.get_text(strip=True)
                break
        data["price"] = clean_price(price_text)
    except Exception:
        data["warnings"].append("price selector failed")

    # Category — breadcrumb second-to-last item
    try:
        breadcrumb = soup.select_one(AMAZON_SELECTORS["breadcrumb"])
        if breadcrumb:
            items = breadcrumb.select("a")
            if len(items) >= 2:
                data["category"] = items[-2].get_text(strip=True)
            elif items:
                data["category"] = items[-1].get_text(strip=True)
    except Exception:
        data["warnings"].append("category selector failed")

    # Quantity
    data["quantity"] = extract_quantity_from_title(data.get("name", ""))

    return data


def extract_from_flipkart(soup, url) -> dict:
    """Scrape a Flipkart product page."""
    data = {"warnings": []}

    # Brand — breadcrumb second item, or first word of title
    try:
        breadcrumbs = soup.select(FLIPKART_SELECTORS["breadcrumb"])
        if len(breadcrumbs) >= 2:
            data["brand"] = breadcrumbs[1].get_text(strip=True)
        else:
            title_el = soup.select_one(FLIPKART_SELECTORS["title"])
            if title_el:
                data["brand"] = title_el.get_text(strip=True).split()[0]
    except Exception:
        data["warnings"].append("brand selector failed")

    # Title
    try:
        title_el = soup.select_one(FLIPKART_SELECTORS["title"])
        data["name"] = title_el.get_text(strip=True) if title_el else ""
    except Exception:
        data["warnings"].append("title selector failed")

    # Price
    try:
        price_el = soup.select_one(FLIPKART_SELECTORS["price"])
        data["price"] = clean_price(price_el.get_text(strip=True)) if price_el else 0
    except Exception:
        data["warnings"].append("price selector failed")

    # Category — breadcrumb second-to-last
    try:
        breadcrumbs = soup.select(FLIPKART_SELECTORS["breadcrumb"])
        if len(breadcrumbs) >= 2:
            data["category"] = breadcrumbs[-2].get_text(strip=True)
    except Exception:
        data["warnings"].append("category selector failed")

    # Quantity
    data["quantity"] = extract_quantity_from_title(data.get("name", ""))

    return data


def extract_from_myntra(soup, url) -> dict:
    """Scrape a Myntra product page."""
    data = {"warnings": []}

    # Brand
    try:
        brand_el = soup.select_one(MYNTRA_SELECTORS["brand"])
        data["brand"] = brand_el.get_text(strip=True) if brand_el else ""
    except Exception:
        data["warnings"].append("brand selector failed")

    # Title
    try:
        title_el = soup.select_one(MYNTRA_SELECTORS["title"])
        data["name"] = title_el.get_text(strip=True) if title_el else ""
    except Exception:
        data["warnings"].append("title selector failed")

    # Price
    try:
        price_el = soup.select_one(MYNTRA_SELECTORS["price"])
        data["price"] = clean_price(price_el.get_text(strip=True)) if price_el else 0
    except Exception:
        data["warnings"].append("price selector failed")

    # Category — parse from URL slug (Myntra URLs: /brand/product/category/)
    try:
        parts = [p for p in url.split("/") if p]
        if len(parts) >= 4:
            data["category"] = parts[3].replace("-", " ").title()
    except Exception:
        data["warnings"].append("category selector failed")

    # Quantity
    data["quantity"] = extract_quantity_from_title(data.get("name", ""))

    return data


def extract_from_hamaramall(soup, url) -> dict:
    """Scrape a Hamara Mall product page. Selectors TBD — needs live inspection."""
    data = {"warnings": []}

    # Attempt generic extraction using common patterns
    try:
        # Try common title selectors
        for sel in ["h1.product-title", "h1.product-name", "h1", ".product-title", ".product-name"]:
            el = soup.select_one(sel)
            if el and el.get_text(strip=True):
                data["name"] = el.get_text(strip=True)
                break
        if not data.get("name"):
            data["warnings"].append("title selector failed — needs live inspection")
    except Exception:
        data["warnings"].append("title selector failed")

    # Try common price selectors
    try:
        for sel in [".product-price", ".price", ".current-price", "span.price"]:
            el = soup.select_one(sel)
            if el:
                data["price"] = clean_price(el.get_text(strip=True))
                break
    except Exception:
        data["warnings"].append("price selector failed")

    # Brand — first word of title as fallback
    if data.get("name") and not data.get("brand"):
        data["brand"] = data["name"].split()[0]

    # Quantity
    data["quantity"] = extract_quantity_from_title(data.get("name", ""))

    return data


def build_search_query(product: dict) -> str:
    """Combine brand + core name + quantity into a search string (max 60 chars)."""
    parts = []
    if product.get("brand"):
        parts.append(product["brand"])
    if product.get("keywords"):
        parts.extend(product["keywords"][:5])  # Limit to top keywords
    if product.get("quantity"):
        parts.append(product["quantity"])
    query = " ".join(parts)
    return query[:60].strip()
'''

files["searcher.py"] = '''# searcher.py — Cross-platform search

import time
import re
import json
import requests
from urllib.parse import quote_plus
from bs4 import BeautifulSoup

from config import HEADERS, AMAZON_SELECTORS, TOP_N_RESULTS, REQUEST_DELAY


def search_amazon(query: str) -> list:
    """Search Amazon.in and return top results."""
    try:
        url = f"https://www.amazon.in/s?k={quote_plus(query)}"
        resp = requests.get(url, headers=HEADERS["amazon"], timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")

        results = []
        cards = soup.select(AMAZON_SELECTORS["search_result"])
        for card in cards[:TOP_N_RESULTS]:
            title_el = card.select_one(AMAZON_SELECTORS["search_title"])
            price_el = card.select_one(AMAZON_SELECTORS["search_price"])
            link_el = card.select_one(AMAZON_SELECTORS["search_link"])

            if not title_el:
                continue

            title = title_el.get_text(strip=True)
            price_text = price_el.get_text(strip=True) if price_el else "0"
            # Clean price — remove commas, take integer
            price = 0
            try:
                price = int(re.sub(r"[^\\d]", "", price_text))
            except ValueError:
                pass

            prod_url = ""
            if link_el and link_el.get("href"):
                href = link_el["href"]
                prod_url = href if href.startswith("http") else f"https://www.amazon.in{href}"

            results.append({"title": title, "price": price, "url": prod_url})

        return results
    except Exception:
        return []


def search_flipkart(query: str) -> list:
    """Search Flipkart and return top results."""
    try:
        url = f"https://www.flipkart.com/search?q={quote_plus(query)}"
        resp = requests.get(url, headers=HEADERS["flipkart"], timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")

        results = []
        # Flipkart search result cards — try multiple known container selectors
        cards = soup.select("div._1AtVbE, div._1xHGtK, div.slAVV4")
        for card in cards[:TOP_N_RESULTS * 2]:  # Over-select, filter later
            title_el = card.select_one("a.IRpwTa, a.s1Q9rs, div._4rR01T, a._2rpwqI")
            price_el = card.select_one("div._30jeq3, div._25b18c")
            link_el = card.select_one("a.IRpwTa, a.s1Q9rs, a._2rpwqI, a._1fQZEK")

            if not title_el:
                continue

            title = title_el.get_text(strip=True)
            price = 0
            if price_el:
                try:
                    price = int(re.sub(r"[^\\d]", "", price_el.get_text(strip=True)))
                except ValueError:
                    pass

            prod_url = ""
            if link_el and link_el.get("href"):
                href = link_el["href"]
                prod_url = href if href.startswith("http") else f"https://www.flipkart.com{href}"

            results.append({"title": title, "price": price, "url": prod_url})

            if len(results) >= TOP_N_RESULTS:
                break

        return results
    except Exception:
        return []


def search_myntra(query: str) -> list:
    """Search Myntra. Tries JSON in script tag first, falls back to HTML."""
    try:
        search_slug = quote_plus(query).replace("+", "-")
        url = f"https://www.myntra.com/{search_slug}"
        resp = requests.get(url, headers=HEADERS["myntra"], timeout=15)
        resp.raise_for_status()

        results = []

        # Try extracting JSON from script tag (window.__myx or __INITIAL_STATE__)
        json_match = re.search(
            r'(?:window\\.__myx\\s*=|window\\.__INITIAL_STATE__\\s*=)\\s*({.+?});?\\s*</script>',
            resp.text, re.DOTALL
        )
        if json_match:
            try:
                data = json.loads(json_match.group(1))
                # Navigate to product list — structure varies
                products = []
                if "searchData" in data:
                    products = data["searchData"].get("results", {}).get("products", [])
                elif "results" in data:
                    products = data["results"].get("products", [])

                for p in products[:TOP_N_RESULTS]:
                    title = f"{p.get('brand', '')} {p.get('productName', '')}".strip()
                    price = p.get("price", 0) or p.get("discountedPrice", 0)
                    prod_url = f"https://www.myntra.com/{p.get('landingPageUrl', '')}"
                    results.append({"title": title, "price": int(price), "url": prod_url})

                if results:
                    return results
            except (json.JSONDecodeError, KeyError):
                pass

        # Fallback: parse HTML
        soup = BeautifulSoup(resp.text, "lxml")
        cards = soup.select(".product-base, .results-base li")
        for card in cards[:TOP_N_RESULTS]:
            brand_el = card.select_one(".product-brand")
            name_el = card.select_one(".product-product")
            price_el = card.select_one(".product-discountedPrice, .product-price")
            link_el = card.select_one("a")

            brand = brand_el.get_text(strip=True) if brand_el else ""
            name = name_el.get_text(strip=True) if name_el else ""
            title = f"{brand} {name}".strip()
            if not title:
                continue

            price = 0
            if price_el:
                try:
                    price = int(re.sub(r"[^\\d]", "", price_el.get_text(strip=True)))
                except ValueError:
                    pass

            prod_url = ""
            if link_el and link_el.get("href"):
                href = link_el["href"]
                prod_url = href if href.startswith("http") else f"https://www.myntra.com{href}"

            results.append({"title": title, "price": price, "url": prod_url})

        return results
    except Exception:
        return []


def search_hamaramall(query: str) -> list:
    """Search Hamara Mall. URL pattern and selectors need live verification."""
    try:
        url = f"https://www.hamaramall.com/search?q={quote_plus(query)}"
        resp = requests.get(url, headers=HEADERS["hamaramall"], timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")

        results = []
        # Try common e-commerce search result patterns
        cards = soup.select(".product-card, .product-item, .search-result-item, .product-grid-item")
        for card in cards[:TOP_N_RESULTS]:
            title_el = card.select_one("h2, h3, .product-title, .product-name, a.product-link")
            price_el = card.select_one(".price, .product-price, .current-price")
            link_el = card.select_one("a")

            if not title_el:
                continue

            title = title_el.get_text(strip=True)
            price = 0
            if price_el:
                try:
                    price = int(re.sub(r"[^\\d]", "", price_el.get_text(strip=True)))
                except ValueError:
                    pass

            prod_url = ""
            if link_el and link_el.get("href"):
                href = link_el["href"]
                prod_url = href if href.startswith("http") else f"https://www.hamaramall.com{href}"

            results.append({"title": title, "price": price, "url": prod_url})

        return results
    except Exception:
        return []


def search_all_platforms(product: dict, skip_platform: str) -> dict:
    """Search all platforms except the source. Returns dict of platform -> results list."""
    search_funcs = {
        "amazon": search_amazon,
        "flipkart": search_flipkart,
        "myntra": search_myntra,
        "hamaramall": search_hamaramall,
    }

    query = product.get("search_query", "")
    results = {}

    for platform, func in search_funcs.items():
        if platform == skip_platform:
            continue
        results[platform] = func(query)
        time.sleep(REQUEST_DELAY)

    return results
'''

files["matcher.py"] = '''# matcher.py — Scoring & match selection

import time
from config import SCORE_EXACT, SCORE_SIMILAR, REQUEST_DELAY
from utils import normalize_quantity, extract_quantity_from_title


def score_result(source: dict, result: dict) -> int:
    """Score a single search result against the source product (max 100)."""
    score = 0

    # Rule 1 — Brand match (required gate, +40)
    if source.get("brand") and source["brand"].lower() not in result.get("title", "").lower():
        return 0
    score += 40

    # Rule 2 — Quantity match (+35)
    source_qty = normalize_quantity(source.get("quantity", ""))
    result_qty = extract_quantity_from_title(result.get("title", ""))
    if source_qty and result_qty and source_qty == normalize_quantity(result_qty):
        score += 35

    # Rule 3 — Keyword overlap (+25)
    source_kw = set(source.get("keywords", []))
    result_words = set(result.get("title", "").lower().split())
    if source_kw:
        overlap = len(source_kw & result_words) / len(source_kw)
        score += int(overlap * 25)

    return score


def pick_best_match(source: dict, results: list) -> dict:
    """Score all results, return best with score and match_type."""
    if not results:
        return {"match_type": "not_found", "score": 0}

    best = None
    best_score = -1

    for r in results:
        s = score_result(source, r)
        if s > best_score:
            best_score = s
            best = r

    if best is None:
        return {"match_type": "not_found", "score": 0}

    match = dict(best)
    match["score"] = best_score
    if best_score >= SCORE_EXACT:
        match["match_type"] = "exact"
    elif best_score >= SCORE_SIMILAR:
        match["match_type"] = "similar"
    else:
        match["match_type"] = "not_found"

    return match


def fallback_search(source: dict, platform: str) -> dict:
    """Re-search with relaxed queries when best score is not_found."""
    from searcher import search_amazon, search_flipkart, search_myntra, search_hamaramall

    search_funcs = {
        "amazon": search_amazon,
        "flipkart": search_flipkart,
        "myntra": search_myntra,
        "hamaramall": search_hamaramall,
    }
    func = search_funcs.get(platform)
    if not func:
        return {"match_type": "unavailable"}

    brand = source.get("brand", "")
    keywords = source.get("keywords", [])

    # Fallback 1: brand + product name (no quantity)
    query1 = f"{brand} {' '.join(keywords[:5])}".strip()
    if query1:
        time.sleep(REQUEST_DELAY)
        results = func(query1)
        match = pick_best_match(source, results)
        if match["match_type"] != "not_found":
            return match

    # Fallback 2: product type + key ingredient (no brand)
    query2 = " ".join(keywords[:3]).strip()
    if query2:
        time.sleep(REQUEST_DELAY)
        results = func(query2)
        match = pick_best_match(source, results)
        if match["match_type"] != "not_found":
            return match

    return {"match_type": "unavailable"}


def match_all_platforms(source: dict, search_results: dict) -> dict:
    """For each platform, pick best match; fallback if not_found."""
    matches = {}
    for platform, results in search_results.items():
        match = pick_best_match(source, results)
        if match["match_type"] == "not_found":
            match = fallback_search(source, platform)
        matches[platform] = match
    return matches
'''

files["sheets.py"] = '''# sheets.py — Google Sheets commission lookup

import gspread
from google.oauth2.service_account import Credentials

from config import SHEET_NAME, SHEET_TAB, COMMISSION_COL_CATEGORY, COMMISSION_COL_PERCENT

# Module-level cache
_commission_cache = None


def load_commission_sheet() -> dict:
    """Authenticate with gspread, read commission sheet, return {category: percent} dict."""
    global _commission_cache
    if _commission_cache is not None:
        return _commission_cache

    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly",
    ]
    creds = Credentials.from_service_account_file(
        "service_account.json", scopes=scopes
    )
    client = gspread.authorize(creds)

    sheet = client.open(SHEET_NAME)
    tab = sheet.worksheet(SHEET_TAB)
    rows = tab.get_all_values()

    commission_map = {}
    for row in rows[1:]:  # Skip header
        if len(row) > max(COMMISSION_COL_CATEGORY, COMMISSION_COL_PERCENT):
            category = row[COMMISSION_COL_CATEGORY].strip()
            try:
                percent = float(row[COMMISSION_COL_PERCENT].strip().replace("%", ""))
            except (ValueError, IndexError):
                percent = 0.0
            if category:
                commission_map[category] = percent

    _commission_cache = commission_map
    return _commission_cache


def get_commission(category: str, commission_map: dict) -> float:
    """Look up commission % for a category. Tries exact, case-insensitive, then partial match."""
    if not category or not commission_map:
        return 0.0

    # Exact match
    if category in commission_map:
        return commission_map[category]

    # Case-insensitive match
    cat_lower = category.lower()
    for key, val in commission_map.items():
        if key.lower() == cat_lower:
            return val

    # Partial match
    for key, val in commission_map.items():
        if cat_lower in key.lower() or key.lower() in cat_lower:
            return val

    return 0.0


def refresh_commission_cache():
    """Clear cache and reload from sheet."""
    global _commission_cache
    _commission_cache = None
    return load_commission_sheet()
'''

files["formatter.py"] = '''# formatter.py — Telegram message builder

from config import PLATFORM_EMOJI, SCORE_EXACT


def format_platform_row(platform: str, match: dict, source_platform: str) -> str:
    """Build one line of the price comparison table."""
    emoji = PLATFORM_EMOJI.get(platform, "❓")
    name = platform.capitalize()
    if platform == "hamaramall":
        name = "Hamara Mall"

    suffix = " (your link)" if platform == source_platform else ""

    match_type = match.get("match_type", "unavailable")
    if match_type == "unavailable":
        return f"{emoji} {name:<12} —    ❌ not listed"
    if match_type == "not_found":
        return f"{emoji} {name:<12} —    ❌ not found"

    price = match.get("price", 0)
    price_str = f"₹{price}" if price else "—"

    if match_type == "exact":
        indicator = "✅"
    elif match_type == "similar":
        indicator = "⚠️ similar"
    else:
        indicator = "❌"

    return f"{emoji} {name:<12} {price_str:<8} {indicator}{suffix}"


def find_best_deal(matches: dict) -> tuple:
    """Find cheapest platform and best earning platform among exact matches.
    Returns (cheapest_platform, best_earning_platform)."""
    cheapest_platform = None
    cheapest_price = float("inf")
    best_earning_platform = None
    best_earning = 0.0

    for platform, match in matches.items():
        if match.get("match_type") != "exact":
            continue
        price = match.get("price", 0)
        if price and price < cheapest_price:
            cheapest_price = price
            cheapest_platform = platform

    return cheapest_platform, cheapest_price


def format_full_message(source: dict, matches: dict, commission: float) -> str:
    """Assemble the complete Telegram message."""
    brand = source.get("brand", "")
    name = source.get("name", "")
    quantity = source.get("quantity", "")
    category = source.get("category", "")
    source_platform = source.get("platform", "")

    # Header
    lines = [
        f"🔍 {brand} {name} {quantity}".strip(),
        f"📂 {category} | 💸 EarnKaro Commission: {commission}%" if commission else f"📂 {category}",
        "",
        "💰 Price Comparison",
        "━━━━━━━━━━━━━━━━━━━━",
    ]

    # Source platform row first
    if source_platform:
        source_match = {
            "match_type": "exact",
            "price": source.get("price", 0),
        }
        lines.append(format_platform_row(source_platform, source_match, source_platform))

    # Other platforms
    platform_order = ["amazon", "flipkart", "myntra", "hamaramall"]
    for p in platform_order:
        if p == source_platform:
            continue
        if p in matches:
            lines.append(format_platform_row(p, matches[p], source_platform))

    # Best deal
    cheapest_platform, cheapest_price = find_best_deal(matches)
    # Also check source price
    source_price = source.get("price", 0)
    if source_price and (not cheapest_platform or source_price < cheapest_price):
        cheapest_platform = source_platform
        cheapest_price = source_price

    if cheapest_platform:
        lines.append("")
        p_name = "Hamara Mall" if cheapest_platform == "hamaramall" else cheapest_platform.capitalize()
        lines.append(f"🏆 Lowest price: {p_name} ₹{cheapest_price}")

    # Best earning
    if commission and cheapest_platform:
        earning = round(cheapest_price * commission / 100, 2)
        lines.append(f"💡 Best earning: {p_name} → ₹{earning}")

    # Similar warning
    has_similar = any(m.get("match_type") == "similar" for m in matches.values())
    if has_similar:
        lines.append("")
        lines.append("⚠️ Note: Similar products may differ in size/variant")

    return "\\n".join(lines)
'''

files["bot.py"] = '''# bot.py — Main Telegram bot entry point

import os
import traceback
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

from utils import detect_platform
from extractor import extract_product
from searcher import search_all_platforms
from matcher import match_all_platforms
from sheets import load_commission_sheet, get_commission
from formatter import format_full_message

# Module-level commission map — loaded at startup
commission_map = {}

START_MESSAGE = """👋 Welcome to EarnKaro Price Bot!

Send me any product link from:
- Amazon.in
- Flipkart
- Myntra
- Hamara Mall

I'll compare prices across all platforms and show your EarnKaro commission instantly."""


async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handle /start command."""
    await update.message.reply_text(START_MESSAGE)


async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Main handler — runs on every user message."""
    url = update.message.text.strip()

    # Step 1 — Validate it's a known platform URL
    try:
        platform = detect_platform(url)
    except ValueError:
        await update.message.reply_text(
            "⚠️ Please send a product link from Amazon, Flipkart, Myntra, or Hamara Mall."
        )
        return

    # Step 2 — Acknowledge immediately
    await update.message.reply_text("⏳ Searching across platforms, give me a moment...")

    try:
        # Step 3 — Extract product from source URL
        product = extract_product(url)

        if not product.get("name"):
            await update.message.reply_text(
                "❌ Couldn't extract product details from that link. Please try a different one."
            )
            return

        # Step 4 — Search all other platforms
        search_results = search_all_platforms(product, skip_platform=platform)

        # Step 5 — Match and score
        matches = match_all_platforms(product, search_results)

        # Step 6 — Get commission
        commission = get_commission(product.get("category", ""), commission_map)

        # Step 7 — Format and send
        message = format_full_message(product, matches, commission)
        await update.message.reply_text(message)

    except Exception as e:
        traceback.print_exc()
        await update.message.reply_text(
            "❌ Something went wrong. Please try again or send a different link."
        )


async def error_handler(update: object, context: ContextTypes.DEFAULT_TYPE):
    """Catch all unhandled exceptions."""
    print(f"Error: {context.error}")
    traceback.print_exc()


def main():
    """Load commission map, build bot, start polling."""
    global commission_map

    # Load .env file if present
    env_path = os.path.join(os.path.dirname(__file__) or ".", ".env")
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, val = line.split("=", 1)
                    os.environ.setdefault(key.strip(), val.strip())

    token = os.environ.get("TELEGRAM_BOT_TOKEN", "")
    if not token:
        print("❌ TELEGRAM_BOT_TOKEN not set. Use the token setup cell or create a .env file.")
        return

    # Load commissions
    try:
        commission_map = load_commission_sheet()
        print(f"✅ Loaded {len(commission_map)} commission categories")
    except Exception as e:
        print(f"⚠️ Could not load commission sheet: {e}")
        print("   Bot will run without commission data.")

    app = Application.builder().token(token).build()

    # Register handlers
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    app.add_error_handler(error_handler)

    print("🤖 Bot is running... Send it a product link on Telegram")
    app.run_polling()


if __name__ == "__main__":
    main()
'''

for fname, content in files.items():
    with open(fname, "w") as f:
        f.write(content)
    print(f"  ✅ {fname}")

print(f"\n✅ All {len(files)} files written to {os.getcwd()}")

  ✅ config.py
  ✅ utils.py
  ✅ extractor.py
  ✅ searcher.py
  ✅ matcher.py
  ✅ sheets.py
  ✅ formatter.py
  ✅ bot.py

✅ All 8 files written to /content/drive/MyDrive/earnkaro_bot


In [12]:
# === Cell 4: Upload service_account.json (Google Sheets Auth) ===
# Option A: Upload from your computer
from google.colab import files as colab_files
import os

if not os.path.exists("service_account.json"):
    print("📤 Upload your service_account.json file:")
    uploaded = colab_files.upload()
    if "service_account.json" in uploaded:
        print("✅ service_account.json uploaded")
    else:
        print("⚠️ File not found. Make sure it's named 'service_account.json'")
else:
    print("✅ service_account.json already exists")

print()
print("📌 Don't forget: Share your Google Sheet with the service account email!")
print("   (Find the email in service_account.json → 'client_email' field)")

📤 Upload your service_account.json file:


KeyboardInterrupt: 

In [9]:
# === Cell 5: Enter Telegram Bot Token ===
import os
from getpass import getpass

token = getpass("Enter your TELEGRAM_BOT_TOKEN: ")
os.environ["TELEGRAM_BOT_TOKEN"] = token
print("✅ Token saved")

8740411923:AAF6lPkvZh_oxniU6vuCCUEvb1B7C_SMDuE··········
✅ Token saved


In [10]:
# === Cell 6: Run Unit Tests ===
# Tests utils, matcher, and end-to-end pipeline (with mock data)


from utils import detect_platform, normalize_quantity, extract_quantity_from_title, clean_price, extract_keywords

# detect_platform
assert detect_platform("https://www.amazon.in/dp/B08XYZ") == "amazon"
assert detect_platform("https://www.flipkart.com/some-product") == "flipkart"
assert detect_platform("https://www.myntra.com/shirts/123") == "myntra"
assert detect_platform("https://www.hamaramall.com/product/456") == "hamaramall"
try:
    detect_platform("https://www.unknown.com/abc")
    assert False, "Should have raised ValueError"
except ValueError:
    pass
print("✅ detect_platform passed")

# normalize_quantity
assert normalize_quantity("250 ML") == "250ml"
assert normalize_quantity("1 KG") == "1000g"
assert normalize_quantity("250ml") == "250ml"
assert normalize_quantity("100gm") == "100g"
assert normalize_quantity("pack of 3") == "3pack"
assert normalize_quantity("x3") == "3pack"
assert normalize_quantity("") == ""
print("✅ normalize_quantity passed")

# extract_quantity_from_title
assert extract_quantity_from_title("Mamaearth Onion Hair Oil 250ml") == "250ml"
assert extract_quantity_from_title("Protein Powder 1kg Pack") == "1kg"
assert extract_quantity_from_title("Face Wash 200g") == "200g"
assert extract_quantity_from_title("Soap pack of 3") == "pack of 3"
assert extract_quantity_from_title("Shampoo 500 ml bottle") == "500 ml"
assert extract_quantity_from_title("No quantity here") == ""
print("✅ extract_quantity_from_title passed")

# clean_price
assert clean_price("\u20b91,299") == 1299
assert clean_price("Rs. 1299.00") == 1299
assert clean_price("\u20b9599") == 599
assert clean_price("") == 0
assert clean_price("abc") == 0
print("✅ clean_price passed")

# extract_keywords
kw = extract_keywords("Mamaearth Onion Hair Oil for Hair Growth & Hair Fall Control 250ml", "Mamaearth")
assert "onion" in kw
assert "hair" in kw
assert "oil" in kw
assert "mamaearth" not in kw
assert "for" not in kw
print("✅ extract_keywords passed")

print("\n🎉 All utils.py tests passed!")



from matcher import score_result, pick_best_match

source = {
    "brand": "Mamaearth",
    "quantity": "250ml",
    "keywords": ["onion", "hair", "oil", "growth", "control"],
}

result_exact = {"title": "Mamaearth Onion Hair Oil 250ml", "price": 299}
result_similar = {"title": "Mamaearth Onion Hair Oil 100ml", "price": 199}
result_different = {"title": "WOW Onion Hair Oil 250ml", "price": 280}

# score_result
s_exact = score_result(source, result_exact)
s_similar = score_result(source, result_similar)
s_different = score_result(source, result_different)
print(f"Scores — exact: {s_exact}, similar: {s_similar}, different brand: {s_different}")

assert s_exact >= 75, f"Expected >=75, got {s_exact}"
assert 40 <= s_similar < 75, f"Expected 40-74, got {s_similar}"
assert s_different == 0, f"Expected 0 (brand mismatch), got {s_different}"
print("✅ score_result passed")

# pick_best_match — should pick exact
best = pick_best_match(source, [result_exact, result_similar, result_different])
assert best["match_type"] == "exact"
assert best["score"] >= 75
print(f"✅ pick_best_match picked: score={best['score']}, type={best['match_type']}")

# pick_best_match — empty list
empty = pick_best_match(source, [])
assert empty["match_type"] == "not_found"
print("✅ pick_best_match empty list → not_found")

# pick_best_match — only similar
best_sim = pick_best_match(source, [result_similar])
assert best_sim["match_type"] == "similar"
print(f"✅ pick_best_match similar only: score={best_sim['score']}, type={best_sim['match_type']}")

# pick_best_match — only brand mismatch
best_diff = pick_best_match(source, [result_different])
assert best_diff["match_type"] == "not_found"
print("✅ pick_best_match brand mismatch → not_found")

print("\n🎉 All matcher.py tests passed!")


# Runs the full pipeline with mock search results to verify integration.
# For live testing, replace mock patches with real URLs.

import unittest.mock as mock
from extractor import extract_product, build_search_query
from searcher import search_all_platforms
from matcher import match_all_platforms
from sheets import get_commission
from formatter import format_full_message

# Mock commission map
commission_map = {
    "Hair Care": 12.0,
    "Skin Care": 10.0,
    "Electronics": 5.0,
}

# --- Test Case 1: Full pipeline with mock data ---
print("=" * 50)
print("TEST 1: Full pipeline (mock data)")
print("=" * 50)

# Simulate extracted product
product = {
    "platform": "amazon",
    "url": "https://www.amazon.in/Mamaearth-Onion-Hair-Oil/dp/B07WLXGKWD",
    "brand": "Mamaearth",
    "name": "Onion Hair Oil for Hair Growth & Hair Fall Control",
    "quantity": "250ml",
    "keywords": ["onion", "hair", "oil", "growth", "control"],
    "category": "Hair Care",
    "price": 299,
    "search_query": "Mamaearth onion hair oil growth control 250ml",
    "warnings": [],
}

# Mock search results
mock_search_results = {
    "flipkart": [
        {"title": "Mamaearth Onion Hair Oil 250ml", "price": 319, "url": "https://flipkart.com/..."},
        {"title": "Mamaearth Onion Shampoo 250ml", "price": 349, "url": "https://flipkart.com/..."},
    ],
    "myntra": [
        {"title": "Mamaearth Onion Hair Oil 100ml", "price": 199, "url": "https://myntra.com/..."},
    ],
    "hamaramall": [],
}

# Run matching
matches = match_all_platforms(product, mock_search_results)
commission = get_commission(product["category"], commission_map)
message = format_full_message(product, matches, commission)

print(message)
print()
assert "Mamaearth" in message
assert "12.0%" in message
assert "✅" in message
print("✅ Test 1 passed\n")

# --- Test Case 2: No matches found ---
print("=" * 50)
print("TEST 2: No matches on any platform")
print("=" * 50)

mock_empty = {
    "flipkart": [],
    "myntra": [],
    "hamaramall": [],
}

with mock.patch("matcher.fallback_search", return_value={"match_type": "unavailable"}):
    matches2 = match_all_platforms(product, mock_empty)
message2 = format_full_message(product, matches2, commission)

print(message2)
print()
assert "not listed" in message2 or "not found" in message2 or "unavailable" in message2.lower()
print("✅ Test 2 passed\n")

# --- Test Case 3: Product with no commission ---
print("=" * 50)
print("TEST 3: Unknown category (0% commission)")
print("=" * 50)

product3 = dict(product)
product3["category"] = "Unknown Category"
commission3 = get_commission(product3["category"], commission_map)
message3 = format_full_message(product3, matches, commission3)

print(message3)
print()
assert commission3 == 0.0
print("✅ Test 3 passed\n")

print("🎉 All end-to-end tests passed!")




✅ detect_platform passed
✅ normalize_quantity passed
✅ extract_quantity_from_title passed
✅ clean_price passed
✅ extract_keywords passed

🎉 All utils.py tests passed!
Scores — exact: 90, similar: 55, different brand: 0
✅ score_result passed
✅ pick_best_match picked: score=90, type=exact
✅ pick_best_match empty list → not_found
✅ pick_best_match similar only: score=55, type=similar
✅ pick_best_match brand mismatch → not_found

🎉 All matcher.py tests passed!
TEST 1: Full pipeline (mock data)
🔍 Mamaearth Onion Hair Oil for Hair Growth & Hair Fall Control 250ml
📂 Hair Care | 💸 EarnKaro Commission: 12.0%

💰 Price Comparison
━━━━━━━━━━━━━━━━━━━━
🟠 Amazon       ₹299     ✅ (your link)
🟡 Flipkart     ₹319     ✅
🔵 Myntra       ₹199     ⚠️ similar
🏪 Hamara Mall  —    ❌ not listed

🏆 Lowest price: Amazon ₹299
💡 Best earning: Amazon → ₹35.88

⚠️ Note: Similar products may differ in size/variant

✅ Test 1 passed

TEST 2: No matches on any platform
🔍 Mamaearth Onion Hair Oil for Hair Growth & Hair Fa

In [11]:
# === Cell 7: Keep-Alive (Prevents Colab Timeout) ===
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
'''))
print("⏰ Keep-alive active — clicking connect every 60 seconds")

<IPython.core.display.Javascript object>

⏰ Keep-alive active — clicking connect every 60 seconds


In [8]:
# === Cell 8: START THE BOT ===
# ⚠️ This cell BLOCKS while the bot runs. You can't run other cells.
# To stop: click the ⏹ stop button on this cell.

import importlib
import config, utils, extractor, searcher, matcher, sheets, formatter, bot

# Reload all modules (picks up any changes)
for mod in [config, utils, extractor, searcher, matcher, sheets, formatter, bot]:
    importlib.reload(mod)

bot.main()

❌ TELEGRAM_BOT_TOKEN not set. Use the token setup cell or create a .env file.


---

## 🔄 Session Restart Instructions

When Colab reconnects after a timeout:

1. **Cell 1** — `pip install` (dependencies don't persist)
2. **Cell 2** — Mount Drive (reconnects to your files)
3. **Cell 5** — Re-enter Telegram token
4. **Cell 8** — Start the bot

> Skip Cell 3 (files are already on Drive) and Cell 4 (service_account.json is already there).

---

## ⚠️ Known Gotchas

- **Flipkart CSS classes change frequently** — update selectors in the `config.py` section of Cell 3
- **Amazon blocks repeated requests** — `REQUEST_DELAY` is set to 1.5s; increase if you get blocked
- **Myntra search may return JSON** — the extractor tries `__INITIAL_STATE__` first
- **Hamara Mall selectors are generic** — inspect a live product page and update `HAMARAMALL_SELECTORS`
- **`run_polling()` blocks** — Cell 8 must be the last cell you run